<a href="https://colab.research.google.com/github/NanC-Nabil/NTI/blob/main/session3_NTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = x_train.reshape(-1, 28, 28, 1)                #(1-Gray scale channel)
x_test = x_test.reshape(-1, 28, 28, 1)

# CNN Model
model = Sequential([         #هنستفتح
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),   #(32-filters no) (3x3-filter size)
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),  #64 filters
    MaxPooling2D((2,2)), #(sq no)

    Flatten(),

    Dense(128, activation='relu'),   #128 neurons
    Dense(10, activation='softmax')    #10 classes
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# Test
loss, acc = model.evaluate(x_test, y_test)

print("Accuracy =", acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


750/750 ━━━━━━━━━━━━━━━━━━━━ 28s 37ms/step - accuracy: 0.9448 - loss: 0.1823 - val_accuracy: 0.9801 - val_loss: 0.0665
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 33ms/step - accuracy: 0.9829 - loss: 0.0545 - val_accuracy: 0.9863 - val_loss: 0.0490
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.9887 - loss: 0.0365 - val_accuracy: 0.9887 - val_loss: 0.0400
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 39s 34ms/step - accuracy: 0.9918 - loss: 0.0266 - val_accuracy: 0.9864 - val_loss: 0.0449
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 41s 33ms/step - accuracy: 0.9941 - loss: 0.0196 - val_accuracy: 0.9872 - val_loss: 0.0458
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9874 - loss: 0.0388
Accuracy = 0.9873999953269958


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    "./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

# CNN Model
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(1,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(

            nn.Flatten(),
            nn.Linear(64*5*5,128),
            nn.ReLU(),
            nn.Linear(128,10)

        )

    def forward(self,x):

        x = self.conv(x)
        x = self.fc(x)

        return x

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(5):

    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

# Testing
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted==labels).sum().item()

print("Accuracy =",100*correct/total,"%")

Epoch 1: Loss = 0.1112
Epoch 2: Loss = 0.0129
Epoch 3: Loss = 0.0072
Epoch 4: Loss = 0.0073
Epoch 5: Loss = 0.0232
Accuracy = 99.01 %
